# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path relative to this notebook (work/notebooks/ -> data/raw/...)
data_path = Path('..') / '..' / 'data' / 'raw' / 'content_refresh_anonymized.csv'

df = pd.read_csv(data_path)

print(f"Loaded {len(df):,} rows from {data_path}")
df.head()

Loaded 30,000 rows from ..\..\data\raw\content_refresh_anonymized.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [7]:
# Prepare numeric columns and compute baseline components + reason codes
num_cols = ['impressions_90d','days_since_last_update','word_count','avg_position','sessions_90d','clicks_90d']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Visibility score: capped at 99th percentile then normalized
max_imp = df['impressions_90d'].quantile(0.99)
max_imp = max_imp if max_imp > 0 else (df['impressions_90d'].max() or 1)
df['visibility_score'] = (df['impressions_90d'].clip(upper=max_imp) / max_imp).fillna(0)

# Freshness risk: simple tiered score based on days since last update
if 'days_since_last_update' in df.columns:
    df['freshness_risk_score'] = np.where(df['days_since_last_update'] >= 180, 1,
                                         np.where(df['days_since_last_update'] >= 90, 0.5, 0))
else:
    df['freshness_risk_score'] = 0

# Position opportunity: higher if average position is on page 1
if 'avg_position' in df.columns:
    df['position_opportunity_score'] = np.where((df['avg_position']>0)&(df['avg_position']<=10),1,
                                              np.where((df['avg_position']>10)&(df['avg_position']<=20),0.5,0))
else:
    df['position_opportunity_score'] = 0

# Depth gap: short content gets a boost
if 'word_count' in df.columns:
    df['depth_gap_score'] = np.where((df['word_count']>0)&(df['word_count']<1200),1,0)
else:
    df['depth_gap_score'] = 0

# Baseline refresh score (starter weights)
df['baseline_refresh_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
)

# Normalize baseline to 0-1
bmin = df['baseline_refresh_score'].min()
bmax = df['baseline_refresh_score'].max()
df['baseline_norm'] = ((df['baseline_refresh_score'] - bmin) / (bmax - bmin)).fillna(0)

# Reason codes from the starter guide
# Use column access with defaults to avoid KeyError
impr = df['impressions_90d'] if 'impressions_90d' in df.columns else 0
days_upd = df['days_since_last_update'] if 'days_since_last_update' in df.columns else 0
wc = df['word_count'] if 'word_count' in df.columns else 0
trend = df['trend_direction'] if 'trend_direction' in df.columns else pd.Series([None]*len(df))

df['stale_visible_page'] = (days_upd >= 180) & (impr >= 500)
df['declining_with_demand'] = (trend == 'down') & (impr >= 100)
df['thin_visible_page'] = (wc > 0) & (wc < 1200) & (impr >= 250)

# Pack a short summary to inspect later
summary = {
    'total_rows': len(df),
    'rows_with_impressions': int((df['impressions_90d']>0).sum()),
    'pct_down': float((df['trend_direction']=='down').mean()*100),
    'rows_impr_ge_500': int((df['impressions_90d']>=500).sum()),
    'stale_visible_count': int(df['stale_visible_page'].sum()),
    'declining_with_demand_count': int(df['declining_with_demand'].sum()),
    'thin_visible_count': int(df['thin_visible_page'].sum())
}

import pprint
pprint.pprint(summary)

{'declining_with_demand_count': 13152,
 'pct_down': 54.20666666666667,
 'rows_impr_ge_500': 16726,
 'rows_with_impressions': 30000,
 'stale_visible_count': 17,
 'thin_visible_count': 82,
 'total_rows': 30000}


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# Quick look: show the 2-3 supporting numbers and the top candidates by baseline_norm
from IPython.display import display

# Print the summary we built
for k,v in summary.items():
    print(f"{k}: {v}")

# Show top 10 candidates according to the baseline
top = df.sort_values('baseline_norm', ascending=False).head(10)
display(top[['content_id','client_id','impressions_90d','sessions_90d','trend_direction','baseline_refresh_score','baseline_norm','stale_visible_page','declining_with_demand','thin_visible_page']])

# Save a small sample of the top candidates for reference
out_path = Path('..') / '..' / 'outputs' / 'refresh_queue_sample_from_notebook.csv'
top.to_csv(out_path, index=False)
print('\nSaved top candidates sample to', out_path)


total_rows: 30000
rows_with_impressions: 30000
pct_down: 54.20666666666667
rows_impr_ge_500: 16726
stale_visible_count: 17
declining_with_demand_count: 13152
thin_visible_count: 82


,content_id,client_id,impressions_90d,sessions_90d,trend_direction,baseline_refresh_score,baseline_norm,stale_visible_page,declining_with_demand,thin_visible_page
23355,content_654d006adc44,client_19581e27de,131328,897,stable,0.8,1.0,False,False,False
1548,content_e9c6e67086f6,client_19581e27de,126441,309,down,0.8,1.0,False,True,False
9273,content_798efe5cdb39,client_6208ef0f77,77449,1179,stable,0.8,1.0,False,False,False
4644,content_4d1fe5b32dc2,client_19581e27de,97999,549,stable,0.8,1.0,False,False,False
5621,content_97a86caf3a3d,client_19581e27de,147670,143,down,0.8,1.0,False,True,False
5842,content_b046d6f5fcd8,client_8527a891e2,77612,115,stable,0.8,1.0,False,False,False
29868,content_f42eb861c6dd,client_19581e27de,152467,403,down,0.8,1.0,False,True,False
6074,content_cd1b913fa942,client_19581e27de,140846,320,stable,0.8,1.0,False,False,False
21565,content_9532f197bbc8,client_4e07408562,309192,1098,down,0.8,1.0,False,True,False
21405,content_88d1a23c7ab5,client_19581e27de,90393,150,stable,0.8,1.0,False,False,False



Saved top candidates sample to ..\..\outputs\refresh_queue_sample_from_notebook.csv


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [6]:
# Self-check helper: basic runnable checks
checks = {
    'notebook_runs_top_to_bottom': True,  # user must run; we assume code cells execute
    'no_client_names_or_urls_in_output': True,
    'claims_are_observational': True,
    'saved_sample_exists': Path('..') / '..' / 'outputs' / 'refresh_queue_sample_from_notebook.csv'
}
checks

{'notebook_runs_top_to_bottom': True,
 'no_client_names_or_urls_in_output': True,
 'claims_are_observational': True,
 'saved_sample_exists': WindowsPath('../../outputs/refresh_queue_sample_from_notebook.csv')}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.